# sberchall — Kaggle runner

This notebook is a **stub on purpose**. It clones the repo and runs its entry point:

```
python -m src
```

Nothing else lives here. What a run *does* — the configuration, the stages, the reporting —
is `src/experiment.py` in the repo. So the loop is: commit to `main`, press **Run all**
here, read the log. This notebook should not need to change between experiments.

Runs unchanged on **Kaggle** and **Google Colab**.

## Settings

- **Kaggle** — Settings -> Accelerator -> GPU (P100 or T4), and Settings -> Internet -> On.
  Both are required; the clone is a network call and training is not CPU-feasible.
- **Colab** — Runtime -> Change runtime type -> GPU. Internet is already on.

The repo is public, so the clone is anonymous — no Kaggle secret, no SSH key, no token.
`J.npy` and `h_train.npy` are tracked in git, so no dataset needs attaching.

Artefacts (`best.pt`, `submission_train.csv`, `summary.json`, ...) are written to
`/kaggle/working` on Kaggle (the output pane) and to `/content/<name>` on Colab (the file
browser). Colab runtimes are ephemeral — download anything you want to keep.

In [ ]:
REPO   = "https://github.com/brkdrd/sberchall.git"
BRANCH = "main"          # point at a branch to try an experiment before merging it

import shutil, subprocess, sys
from pathlib import Path

# Clone outside /kaggle/working: the checkout is not an output, only what src/ writes is.
SRC = Path("/tmp/sberchall")
if SRC.exists():
    shutil.rmtree(SRC)

clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)],
                       capture_output=True, text=True)
if clone.returncode:
    raise RuntimeError(clone.stderr + "\n\nClone failed. Check, in order:\n"
                       "  1. on Kaggle, Settings -> Internet must be ON (off gives a\n"
                       "     DNS/timeout error); Colab has internet on by default;\n"
                       "  2. the repo is still public (403 / 'Authentication failed' if not);\n"
                       f"  3. the branch '{BRANCH}' exists.")
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--pretty=cloned %h %s"],
                     capture_output=True, text=True).stdout)

# Colab and Kaggle both ship torch/numpy/scipy, so this is normally a no-op — it is here
# so the notebook is genuinely self-installing on a bare runtime.
try:
    import numpy, scipy, torch                                            # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(SRC / "requirements.txt")], check=True)

# Everything from here on is the repo's code. -u so the log streams into this cell live.
proc = subprocess.Popen([sys.executable, "-u", "-m", "src"], cwd=str(SRC),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc:
    raise RuntimeError(f"python -m src exited with code {rc} (see the log above)")
print("\n[runner] done")